In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: QedmaによるQiskit Function
*[APIリファレンス](https://docs.quantum.ibm.com/api/functions/qedma-qesem)をご覧ください*

> **Note:** Qiskit Functionsは、IBM Quantum&reg; Premiumプラン、Flexプラン、およびOn-Prem（IBM Quantum Platform API経由）プランのユーザーのみが利用できる実験的な機能です。プレビューリリースの状態であり、変更される場合があります。

## 概要
近年、量子処理ユニット（QPU）は大幅に改善されていますが、既存のハードウェアにおけるノイズや不完全性によるエラーは、量子アルゴリズム開発者にとって依然として中心的な課題です。この分野が古典的に検証できないユーティリティスケールの量子計算に近づくにつれて、精度を保証しながらノイズをキャンセルするソリューションがますます重要になっています。この課題を克服するため、QedmaはQuantum Error Mitigation（QESEM）を開発しました。これは[Qiskit Function](/guides/functions)として、IBM Quantum Platformにシームレスに統合されています。

QESEMを使用することで、ユーザーはノイズの多いQPU上で量子回路を実行し、基本的な限界に近い非常に効率的なQPU時間オーバーヘッドで、高精度なエラーのない結果を得ることができます。これを実現するために、QESEMはQedmaが開発した独自の手法のスイートを活用して、エラーの特性評価と低減を行っています。エラー低減技術には、ゲート最適化、ノイズを考慮したトランスパイレーション、エラー抑制（ES）、および不偏エラー緩和（EM）が含まれます。これらの特性評価に基づく手法を組み合わせることで、ユーザーは汎用的な大容量量子回路に対して信頼性の高いエラーのない結果を達成でき、それ以外では実現できないアプリケーションを解き放ちます。

基礎となるコンポーネントの詳細な説明とユーティリティスケールのデモについては、論文[Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997)を参照してください。
## 説明
QedmaのQESEM functionを使用することで、エラー抑制と緩和を適用した回路の見積もりと実行を簡単に行い、より大きな回路容量とより高い精度を達成できます。QESEMを使用するには、量子回路、測定するオブザーバブルのセット、各オブザーバブルの目標統計精度、および選択したQPUを指定します。目標精度まで回路を実行する前に、回路の実行を必要としない解析的計算に基づいて必要なQPU時間を見積もることができます。QPU時間の見積もりに納得したら、QESEMで回路を実行できます。

回路を実行すると、QESEMはお使いの回路に合わせたデバイス特性評価プロトコルを実行し、回路内で発生するエラーの信頼性の高いノイズモデルを生成します。この特性評価に基づいて、QESEMはまずノイズを考慮したトランスパイレーションを実装し、入力回路を物理量子ビットとゲートのセットにマッピングして、目標オブザーバブルに影響するノイズを最小化します。これには、ネイティブで使用可能なゲート（IBM&reg;デバイスのCX/CZ）と、QESEMによって最適化された追加のゲートが含まれ、QESEMの拡張ゲートセットを形成しています。次に、QESEMはQPU上で特性評価に基づくESおよびEM回路のセットを実行し、その測定結果を収集します。これらは古典的な後処理によって、要求された精度に対応する各オブザーバブルの不偏期待値とエラーバーを提供します。

![Qedma QESEM overview](../docs/images/guides/qedma-qesem/overview.svg)
QESEMは、さまざまな量子アプリケーションに対して高精度な結果を提供し、現在達成可能な最大の回路容量での実証がなされています。QESEMは以下のユーザー向け機能を提供しており、以下のベンチマークセクションで実証されています：
-	**精度の保証:** QESEMはオブザーバブルの期待値に対する不偏推定を出力します。そのEM手法は理論的な保証を備えており、Qedmaの最先端の特性評価と組み合わせることで、緩和がユーザー指定の精度までノイズのない回路出力に収束することを保証します。系統的なエラーや偏りが生じやすい多くのヒューリスティックなEM手法とは対照的に、QESEMの精度保証は、汎用的な量子回路とオブザーバブルにおいて信頼性の高い結果を確保するために不可欠です。
-	**大規模QPUへのスケーラビリティ:** QESEMのQPU時間は回路容量に依存しますが、それ以外は量子ビット数に依存しません。Qedmaは、IBM Quantum 127量子ビットEagleおよび133量子ビットHeronデバイスを含む、現在利用可能な最大の量子デバイスでQESEMを実証しています。
-	**アプリケーション非依存:** QESEMは、ハミルトニアンシミュレーション、VQE、QAOA、振幅推定など、さまざまなアプリケーションで実証されています。ユーザーは任意の量子回路と測定するオブザーバブルを入力し、正確なエラーのない結果を得ることができます。唯一の制限は、アクセス可能な回路容量と出力精度を決定するハードウェア仕様と割り当てられたQPU時間によって決まります。対照的に、多くのエラー低減ソリューションはアプリケーション固有であるか、制御されていないヒューリスティックを含んでおり、汎用的な量子回路とアプリケーションには適用できません。
-  **拡張ゲートセット:** QESEMは分数角度ゲートをサポートし、IBM Quantum HeronおよびEagleデバイスでQedma最適化の分数角度$Rzz(\theta)$ゲートを提供します。この拡張ゲートセットにより、より効率的なコンパイルが可能になり、デフォルトのCX/CZコンパイルと比較して最大2倍の回路容量が利用可能になります。
-	**マルチベースオブザーバブル:** QESEMは、汎用的なハミルトニアンなど、多くの非可換Pauli文字列で構成された入力オブザーバブルをサポートします。測定ベースの選択とQPUリソース配分（ショットと回路）の最適化は、要求された精度に必要なQPU時間を最小化するためにQESEMによって自動的に実行されます。ハードウェアの忠実度と実行レートを考慮したこの最適化により、より深い回路を実行してより高い精度を得ることができます。
## ベンチマーク
QESEMは、さまざまなユースケースとアプリケーションでテストされています。以下の例は、QESEMで実行できるワークロードの種類を評価するのに役立ちます。

所与の回路とオブザーバブルに対して、エラー緩和と古典的シミュレーションの両方の難しさを定量化するための主要な評価指標は**アクティブボリューム**です：回路内でオブザーバブルに影響するCNOTゲートの数です。アクティブボリュームは、回路の深さと幅、オブザーバブルの重み、およびオブザーバブルのライトコーンを決定する回路構造に依存します。詳細については、[2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s)のトークを参照してください。QESEMは高ボリュームの領域で特に大きな価値を提供し、汎用的な回路とオブザーバブルに対して信頼性の高い結果をもたらします。

![Active volume](../docs/images/guides/qedma-qesem/active_volume.svg)

| アプリケーション    | 量子ビット数 | デバイス | 回路の説明 | 精度 | 合計時間 | ランタイム使用量 |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE回路  | 8              | Eagle (r3) | 21層合計、9測定ベース、1Dチェーン                    | 98%      | 35分      | 14分         |
| Kicked Ising   | 28              | Eagle (r3) | 3ユニーク層 × 3ステップ、2D heavy-hexトポロジー                      | 97%     | 22分      | 4分          |
| Kicked Ising   | 28              | Eagle (r3) | 3ユニーク層 × 8ステップ、2D heavy-hexトポロジー                     | 97%      | 116分      | 23分          |
| トロッタライズドハミルトニアンシミュレーション   | 40  | Eagle (r3)            | 2ユニーク層 × 10トロッターステップ、1Dチェーン                    | 97%      | 3時間     | 25分         |
| トロッタライズドハミルトニアンシミュレーション   | 119  | Eagle (r3)           | 3ユニーク層 × 9トロッターステップ、2D heavy-hexトポロジー                    | 95%      | 6.5時間     | 45分         |
| Kicked Ising  | 136             | Heron (r2) | 3ユニーク層 × 15ステップ、2D heavy-hexトポロジー                    | 99%      | 52分             | 9分           |

ここでの精度はオブザーバブルの理想値に対して相対的に測定されています：$\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$。ここで「$\epsilon$」は緩和の絶対精度（ユーザー入力で設定）であり、$\langle O \rangle_{ideal}$はノイズのない回路でのオブザーバブルです。
「ランタイム使用量」はバッチモードでのベンチマークの使用量を測定します（個々のジョブの使用量の合計）。一方、「合計時間」はセッションモードでの使用量を測定します（実験のウォール時間）。これには追加の古典的処理と通信時間が含まれます。QESEMは両方のモードでの実行が可能であり、ユーザーは利用可能なリソースを最大限に活用できます。

28量子ビットのKicked Ising回路は、ibm_kawasakiの3つの連結ループ上で、Shinjo et al.が研究した離散時間準結晶（[arXiv 2403.16718](https://arxiv.org/abs/2403.16718)および[Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)を参照）をシミュレートします。ここで使用した回路パラメータは$(\theta_x, \theta_z) = (0.9 \pi, 0)$で、強磁性初期状態$| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$を用いています。測定されたオブザーバブルは磁化の絶対値$M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$です。ユーティリティスケールのKicked Ising実験はibm_fezの最良の136量子ビット上で実行されました。この特定のベンチマークはClifford角$(\theta_x, \theta_z) = (\pi, 0)$で実行されました。この角度では、アクティブボリュームが回路深さとともにゆっくりと増加するため、高いデバイス忠実度と合わせて、短いランタイムで高精度が実現できます。

トロッタライズドハミルトニアンシミュレーション回路は、分数角度での横断磁場Isingモデル用のものです：$(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$および$(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$（[Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)を参照）。ユーティリティスケールの回路はibm_brisbaneの最良の119量子ビット上で実行され、40量子ビットの実験は利用可能な最良のチェーン上で実行されました。精度は磁化について報告されており、より高い重みのオブザーバブルについても高精度な結果が得られています。

VQE回路は、Deutsches Elektronen-Synchrotron（DESY）の量子技術・応用センターの研究者と共同で開発されました。ここでの目標オブザーバブルは、多数の非可換Pauli文字列からなるハミルトニアンであり、マルチベースオブザーバブルに対するQESEMの最適化されたパフォーマンスを強調しています。緩和は古典的に最適化されたansatzに適用されました。これらの結果はまだ未発表ですが、同様の構造的特性を持つ異なる回路についても同等の品質の結果が得られます。
## はじめに
[IBM Quantum Platform APIキー](http://quantum.cloud.ibm.com/)を使用して認証し、以下のようにQESEM Qiskit Functionを選択します。（このスニペットは、すでに[アカウントを保存済み](/guides/functions#install-qiskit-functions-catalog-client)であることを前提としています。）

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

使い慣れたQiskit Serverless APIを使用して、Qiskit Functionワークロードのステータスを確認したり、結果を返したりすることができます：

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

以下のコードスニペットは、QPU時間の見積もりを取得する方法を示しています（`estimate_time_only`が設定されている場合）：

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


以下のコードスニペットは、緩和結果（`estimate_time_only`が設定されていない場合）と実行メトリクスを取得する方法を示しています。これらには、さまざまなパラメータがQESEM実行にどのように影響するかをより深く理解するための重要なデータが含まれています。研究に基づいて論文を書く際にも関連する可能性があります。

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## エラーメッセージの取得
ワークロードのステータスがERRORの場合、`job.result()`を使用して以下のようにエラーメッセージを取得してください：

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## サポートを受ける

Qedmaのサポートチームがお手伝いします。QESEM Qiskit Functionの使用中に問題が発生した場合や、ご質問がある場合は、遠慮なくお問い合わせください。知識豊富で親切なサポートスタッフが、技術的な懸念やご質問にお答えする準備ができています。

サポートについては、support@qedma.comまでメールでお問い合わせください。迅速かつ正確な対応ができるよう、お困りの問題についてできるだけ詳しい情報をご提供ください。また、メールまたは電話で担当のQedma POC担当者にご連絡いただくこともできます。

より効率的にサポートするため、お問い合わせの際には以下の情報をご提供ください：

- 問題の詳細な説明
- ジョブID
- 関連するエラーメッセージやコード

私たちは、Qiskit Functionでの最善のご体験をお届けするため、迅速かつ効果的なサポートの提供にコミットしています。

製品の改善に常に取り組んでおり、皆様のご提案を歓迎しています。サービスの強化方法や追加を希望される機能についてアイデアがある場合は、support@qedma.comまでご意見をお送りください。

## 次のステップ

> **Tip:** - [Qedma QESEMへのアクセスをリクエストする](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)。
> - この Qiskit Function の[APIリファレンス](https://docs.quantum.ibm.com/api/functions/qedma-qesem)をご覧ください。
> - レビュー [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199)。
> - レビュー [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997)。